In [1]:
# IMPORTS
from graph import app 
import pandas as pd
import os
from tqdm.auto import tqdm
import torch
import time 
from Configuration.models import models #import models to evaluate
from dotenv import load_dotenv
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI
from langchain_neo4j import Neo4jGraph  # for schema
from langchain_ollama import ChatOllama

In [ ]:
# RUN ONCE 

import nltk

# Dowloading these for NLTK
print("Downloading NLTK data...")
nltk.download('stopwords')
nltk.download('punkt')      
nltk.download('punkt_tab')  
print("Done!")

In [9]:
# CLEANING UP COGNEE 
import cognee

# Remove all user data (datasets, documents)
await cognee.prune.prune_data()

# Remove all system data (graph, vectors, metadata, cache)
await cognee.prune.prune_system(metadata=True)


2026-05-07T13:26:34.634565 [info     ] Database deleted successfully. [cognee.shared.logging_utils]

Storage manager absolute path: C:\Users\olgag\Desktop\agentic-memory\venv\Lib\site-packages\cognee\.cognee_cache

Deleting cache...             

✓ Cache deleted successfully! 


In [4]:
# Loading .env
load_dotenv(override=True) 

print("System Initialization")

# Key Checks
if not os.getenv("OPENAI_API_KEY"):
    print("WARNING: OPENAI_API_KEY missing.")
elif not os.getenv("HUGGINGFACE_API_KEY"):
    print("ERROR: HUGGINGFACE_API_KEY missing.")
elif not os.getenv("NEO4J_URI"):
    print("ERROR: NEO4J URI missing.")
else:
    print("Core API Keys found.")

# Model Settings
model_name = "meta-llama/Meta-Llama-3.1-8B-Instruct"

print(f"\nInitializing Hugging Face model: {model_name}...")

try:
    hf_endpoint = HuggingFaceEndpoint(
        repo_id=model_name, 
        task="text-generation",
        max_new_tokens=512,
        temperature=0.0, 
        huggingfacehub_api_token=os.getenv("HUGGINGFACE_API_KEY")
    )

    model = ChatHuggingFace(llm=hf_endpoint)
    print(f"Model {model_name} loaded successfully.")
    
except Exception as e:
    print(f"Error loading model: {e}")

# Config Settings
config = {
    'configurable': {
        'model': model, 
        'model_name': model_name, 
        'total_attempts': 5,          # total 
        'attempts_w_filtered': 3,     # with filtered
    },
    'recursion_limit': 50
}

# Setting up inputs for next cell
inputs = {
    'database_url': os.getenv("NEO4J_URI"),           
    'database_name': "movies",                        
    'database_user': os.getenv("NEO4J_USERNAME"),    
    'database_password': os.getenv("NEO4J_PASSWORD")  
}

print("\nSetup Complete! Ready for our agent!")

System Initialization

Setup Complete! Ready for our agent!


In [8]:
# THE 10-QUERY CONDENSED DATASET
custom_dataset = [
    {"question": "Find all actors who acted in more than 5 movies", "correct_cypher": "MATCH (a:Person)-[:ACTED_IN]->(m:Movie) WITH a, count(m) AS movie_count WHERE movie_count > 5 RETURN a.name"},
    {"question": "Find movies that have more than 2 actors", "correct_cypher": "MATCH (m:Movie)<-[:ACTED_IN]-(a:Person) WITH m, count(a) AS actor_count WHERE actor_count > 2 RETURN m.title"},
    {"question": "Find actors who acted in movies directed by Tom Hanks", "correct_cypher": "MATCH (a:Person)-[:ACTED_IN]->(m:Movie)<-[:DIRECTED]-(d:Person {name: 'Tom Hanks'}) RETURN DISTINCT a.name"},
    {"question": "Find all directors who directed at least 1 movie", "correct_cypher": "MATCH (d:Person)-[:DIRECTED]->(m:Movie) WITH d, count(m) AS movie_count WHERE movie_count >= 1 RETURN d.name"},
    {"question": "Find all actors who acted in Stand By Me", "correct_cypher": "MATCH (a:Person)-[:ACTED_IN]->(m:Movie {title: 'The Matrix'}) RETURN a.name"}
]

print(f"Loaded condensed dataset with {len(custom_dataset)} queries")

Loaded condensed dataset with 5 queries


In [6]:
# THE 10-QUERY CONDENSED DATASET
custom_dataset = [
    {"question": "Find all actors who acted in more than 3 movies", "correct_cypher": "MATCH (a:Person)-[:ACTED_IN]->(m:Movie) WITH a, count(m) AS movie_count WHERE movie_count > 3 RETURN a.name"},
    {"question": "Find actors and how many distinct movies they acted in", "correct_cypher": "MATCH (a:Person)-[:ACTED_IN]->(m:Movie) RETURN a.name, count(DISTINCT m) AS movie_count"},
    {"question": "Find movies that have more than 3 actors", "correct_cypher": "MATCH (m:Movie)<-[:ACTED_IN]-(a:Person) WITH m, count(a) AS actor_count WHERE actor_count > 3 RETURN m.title"},
    {"question": "Find actors who acted in movies directed by Ron Howard", "correct_cypher": "MATCH (a:Person)-[:ACTED_IN]->(m:Movie)<-[:DIRECTED]-(d:Person {name: 'Ron Howard'}) RETURN DISTINCT a.name"},
    {"question": "Find actors who acted in exactly one movie", "correct_cypher": "MATCH (a:Person)-[:ACTED_IN]->(m:Movie) WITH a, count(m) AS movie_count WHERE movie_count = 1 RETURN a.name"},
    {"question": "Find all directors who directed at least 2 movies", "correct_cypher": "MATCH (d:Person)-[:DIRECTED]->(m:Movie) WITH d, count(m) AS movie_count WHERE movie_count >= 2 RETURN d.name"},
    {"question": "Find all people who both acted in and directed movies", "correct_cypher": "MATCH (p:Person)-[:ACTED_IN]->(:Movie), (p)-[:DIRECTED]->(:Movie) RETURN DISTINCT p.name"},
    {"question": "Find all actors who acted in The Matrix", "correct_cypher": "MATCH (a:Person)-[:ACTED_IN]->(m:Movie {title: 'The Matrix'}) RETURN a.name"},
    {"question": "Find how many movies were released per year", "correct_cypher": "MATCH (m:Movie) WHERE m.released IS NOT NULL RETURN m.released, count(m) AS movie_count"},
    {"question": "Return each movie along with its director", "correct_cypher": "MATCH (m:Movie)<-[:DIRECTED]-(d:Person) RETURN m.title, d.name"}
]

print(f"Loaded condensed dataset with {len(custom_dataset)} queries")

Loaded condensed dataset with 10 queries


In [ ]:
import time
import pandas as pd
from tqdm import tqdm

print("Starting evaluation for custom_dataset on Movies DB...")
evaluation_results = [] 

# 1. CHANGE THE CONFIG NAME FOR THE EXCEL EXPORT
config_name = "CUSTOM_MEMORY" 

for i, data in enumerate(tqdm(custom_dataset, desc="Evaluating Queries")):
    question = data["question"]
    correct_cypher = data["correct_cypher"]
    
    print('\n' + '#' * 60)
    print(f" QUERY {i+1}: {question}")
    print('#' * 60)
    
    inputs['question'] = question
    start_time = time.time()
    
    try:
        # RUN THE LANGGRAPH STATE
        result = app.invoke(inputs, config=config)
        latency = time.time() - start_time
        
        answer = result.get('answer', "No answer provided")
        final_cypher = result.get('cypher_statement', "") 
        
        print('\n' + '='*50)
        print(f"Final Agent Answer: {answer}")
        print('='*50 + '\n')
        
        # EXTRACT METRICS FROM THE STATE
        evaluation_results.append({
            "Configuration": config_name,
            "Question_ID": i + 1,
            "Question": question,
            "Correct_Cypher": correct_cypher,
            "Generated_Cypher": final_cypher,
            "VCR_KG_Valid": result.get("is_valid_cypher", False), 
            "First_Try_Success": result.get("attempts", 1) == 1,
            "Correction_Attempts": result.get("attempts", 1) - 1,
            # Force these to False for the baseline run just to be safe
            "Total_Invocations": result.get("llm_invocations", 1),
            "Total_Tokens": result.get("total_tokens", 0),
            "Latency_Seconds": round(latency, 2),
            "Status": "Success",
            "Error_Log": ""
        })
        
    except Exception as e:
        latency = time.time() - start_time
        print(f"\n Execution Failed: {e}")
        evaluation_results.append({
            "Configuration": config_name,
            "Question_ID": i + 1,
            "Question": question,
            "Correct_Cypher": correct_cypher,
            "Generated_Cypher": "",
            "VCR_KG_Valid": False, 
            "First_Try_Success": False,
            "Correction_Attempts": 0,
            "Total_Invocations": 1,
            "Total_Tokens": 0,
            "Latency_Seconds": round(latency, 2),
            "Status": "Failed",
            "Error_Log": str(e)
        })
    
    # Adding a small buffer to respect API rate limits
    if i < len(custom_dataset) - 1:
        print("Waiting 5s for API rate limits...")
        time.sleep(5)

print("\n Evaluation Complete! Exporting to Excel...")

# EXPORT TO EXCEL
df = pd.DataFrame(evaluation_results)
file_name = f"movies_db_metrics_{config_name}.xlsx"
df.to_excel(file_name, index=False, engine='openpyxl')
print(f"Metrics successfully exported to {file_name}!")

In [ ]:
import json

# --- EVALUATION FUNCTIONS ---

def extract_values(record):
    """Recursively pulls out primitive values from a dictionary, ignoring all keys."""
    values = []
    if isinstance(record, dict):
        for v in record.values():
            if isinstance(v, dict):
                values.extend(extract_values(v))
            else:
                values.append(v)
    else:
        values.append(record)
    return values

def is_sub_record(gt_vals, gen_vals):
    """Checks if all values in a Ground Truth record exist somewhere in the Generated record."""
    temp_gen = list(gen_vals)
    for v in gt_vals:
        if v in temp_gen:
            temp_gen.remove(v) # Safely handle duplicate values
        else:
            return False
    return True

def calculate_accuracy(generated_records, ground_truth_records):
    """
    Execution Accuracy: Validates that every Ground Truth record is satisfied
    by a Generated record (even if the generated record has extra properties).
    """
    # Handle empty returns
    if not generated_records or not ground_truth_records:
        return generated_records == ground_truth_records
        
    # If the queries returned a different number of rows
    if len(generated_records) != len(ground_truth_records):
        return False

    # Extract all values as strings
    gen_rows = [[str(v) for v in extract_values(rec)] for rec in generated_records]
    gt_rows = [[str(v) for v in extract_values(rec)] for rec in ground_truth_records]

    # Map each Ground Truth row to a Generated row
    matched_gen_indices = set()
    for gt_row in gt_rows:
        match_found = False
        for i, gen_row in enumerate(gen_rows):
            if i in matched_gen_indices:
                continue # Row already used
                
            # If the GT values are a subset of the Generated values for this row
            if is_sub_record(gt_row, gen_row):
                matched_gen_indices.add(i)
                match_found = True
                break
                
        if not match_found:
            return False # A ground truth record was missing from the generated results!

    return True


# --- EXECUTION LOGIC ---

file_path = 'baseline_no_memory_results.json' 

try:
    with open(file_path, 'r', encoding='utf-8') as file:
        eval_data = json.load(file)

    total_queries = len(eval_data)
    correct_executions = 0

    print(f"--- RUNNING TEXT2CYPHER EXECUTION ACCURACY EVALUATION ({total_queries} Queries) ---\n")

    for item in eval_data:
        q_id = item['id']
        
        # Calculate Execution Accuracy
        is_accurate = calculate_accuracy(
            item['generated_records'], 
            item['correct_records'] 
        )
        if is_accurate:
            correct_executions += 1
            
        print(f"Query [{q_id}]: Execution Match: {is_accurate}")

    # Final Aggregated Metric
    if total_queries > 0:
        execution_accuracy_percentage = (correct_executions / total_queries) * 100

        print("\n--- FINAL METRICS ---")
        print(f"Execution Accuracy: {execution_accuracy_percentage:.2f}% ({correct_executions}/{total_queries})")
    else:
        print("No queries found in the JSON file.")

except FileNotFoundError:
    print(f"Error: Could not find '{file_path}'. Make sure it is in the same folder as your notebook!")
except json.JSONDecodeError as e:
    print(f"Error: Your JSON file is not formatted correctly. Please check for syntax errors. Details: {e}")

--- RUNNING TEXT2CYPHER EXECUTION ACCURACY EVALUATION (10 Queries) ---

Query [1]: Execution Match: True
Query [2]: Execution Match: True
Query [3]: Execution Match: False
Query [4]: Execution Match: False
Query [5]: Execution Match: False
Query [6]: Execution Match: True
Query [7]: Execution Match: False
Query [8]: Execution Match: True
Query [9]: Execution Match: True
Query [10]: Execution Match: True

--- FINAL METRICS ---
Execution Accuracy: 60.00% (6/10)


CUSTOM MEMORY EXPERIMENT

In [1]:
# IMPORTS
from graph import app 
import pandas as pd
import os
from tqdm.auto import tqdm
import torch
import time 
from Configuration.models import models #import models to evaluate
from dotenv import load_dotenv
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_openai import ChatOpenAI
from langchain_neo4j import Neo4jGraph  # for schema
from langchain_ollama import ChatOllama
from Agent_Memory.memory_manager import log_episode_to_memory

In [4]:
# Loading .env
load_dotenv(override=True) 

print("System Initialization")

# Key Checksss
if not os.getenv("OPENAI_API_KEY"):
    print("ERROR: OPENAI_API_KEY missing.")
elif not os.getenv("HUGGINGFACE_API_KEY"):
    print("ERROR: HUGGINGFACE_API_KEY missing.")
elif not os.getenv("NEO4J_URI"):
    print("ERROR: NEO4J URI missing.")
else:
    print("All API Keys found.")

model_name = "meta-llama/Meta-Llama-3.1-8B-Instruct"

print(f"\nInitializing Hugging Face model: {model_name}...")

try:
    hf_endpoint = HuggingFaceEndpoint(
        repo_id=model_name, 
        task="text-generation",
        max_new_tokens=512,
        temperature=0.1, 
        huggingfacehub_api_token=os.getenv("HUGGINGFACE_API_KEY")
    )

    model = ChatHuggingFace(llm=hf_endpoint)
    print(f"Model {model_name} loaded successfully.")
    
except Exception as e:
    print(f"Error loading model: {e}")

# Config Settings
config = {
    'configurable': {
        'model': model, 
        'model_name': model_name, 
        'total_attempts': 5,          # total 
        'attempts_w_filtered': 3,     # with filtered
    },
    'recursion_limit': 50
}

# Setting up inputs for next cell
inputs = {
    'database_url': os.getenv("NEO4J_URI"),           
    'database_name': "movies",                        
    'database_user': os.getenv("NEO4J_USERNAME"),    
    'database_password': os.getenv("NEO4J_PASSWORD")  
}


print("\nSetup Complete! Ready for our agent!")

System Initialization

Setup Complete! Ready for our agent!


In [ ]:
custom_dataset = [
    "Find all actors who act in more than 3 movies",
    "Find actors and how many distinct movies they are acting in",
    "Find movies that have more than 3 actors"
]

print("Running custom_dataset on Movies DB...")
results_log = [] 

memory_uri = os.getenv("NEO4J_URI")
memory_user = os.getenv("NEO4J_USERNAME")
memory_pw = os.getenv("NEO4J_PASSWORD")

for i, question in enumerate(tqdm(custom_dataset)):
    print('\n' + '#' * 60)
    print(f" QUERY {i+1}: {question}")
    print('#' * 60)
    
    inputs['question'] = question
    
    try:
        result = app.invoke(inputs, config=config)
        
        # Formatting
        print('\n' + '='*50)
        print(f"Final Agent Answer: {result.get('answer', 'No answer generated.')}")
        print('='*50 + '\n')
        
        if i < len(custom_dataset) - 1:
            print("Waiting 10s for API rate limits...")
            time.sleep(10)

    except Exception as e:
        print(f"Execution Failed: {e}")

END OF CUSTOM MEMORY EXPERIMENT

In [ ]:
# Loading .env
load_dotenv(override=True) 

print("System Initialization")

# Key Checksss
if not os.getenv("OPENAI_API_KEY"):
    print("ERROR: OPENAI_API_KEY missing.")
#elif not os.getenv("GROQ_API_KEY"):
    #print("ERROR: GROQ_API_KEY missing. Check your .env file.")
elif not os.getenv("HUGGINGFACE_API_KEY"):
    print("ERROR: HUGGINGFACE_API_KEY missing.")
elif not os.getenv("NEO4J_URI"):
    print("ERROR: NEO4J URI missing.")
else:
    print("All API Keys found.")

# # HYBRID MEMORY Init
# memory = None
# try:
#     from Memory.hybrid_manager import HybridMemoryManager
#     print("\nLoading Hybrid Memory Manager...")
    
#     memory = HybridMemoryManager()
    
#     # Memory Check
#     if memory.cognee:
#         print("Cognee (Semantic): ACTIVE")
#     else:
#         print("Cognee (Semantic): INACTIVE ")
        
#     if memory.graphiti:
#         print("Graphiti Episodic: ACTIVE")
#     else:
#         print("Graphiti Episodic: INACTIVE")

# except ImportError:
#     print("ERROR: Could not import HybridMemoryManager.")
# except Exception as e:
#     print(f"Error initializing Memory: {e}")

# Model Settings
#from langchain_community.chat_models import ChatOllama

#from langchain_openai import ChatOpenAI
#model_name = "llama3.2"
model_name = "meta-llama/Meta-Llama-3.1-8B-Instruct"
#model_name = "meta-llama/Llama-3.2-1B-Instruct"
#model_name = "Qwen/Qwen2.5-3B-Instruct"
#model_name = 'gpt-4o'
#model_name = "llama-3.3-70b-versatile"
#model_name = "llama-3.1-8b-instant"
# model_name = "qwen/qwen3-32b"
#model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
# print(f"\nInitializing Groq model: {model_name}...")

# try:
#     model = ChatGroq(
#         model=model_name,
#         temperature=0.9, # change later?
#         api_key=os.getenv("GROQ_API_KEY")
#     )
#     print(f"Model {model_name} loaded successfully.")
    
# except Exception as e:
#     print(f"Error loading model: {e}")

print(f"\nInitializing Hugging Face model: {model_name}...")

try:
    hf_endpoint = HuggingFaceEndpoint(
        repo_id=model_name, 
        task="text-generation",
        max_new_tokens=512,
        temperature=0.1, 
        huggingfacehub_api_token=os.getenv("HUGGINGFACE_API_KEY")
    )

    model = ChatHuggingFace(llm=hf_endpoint)
    print(f"Model {model_name} loaded successfully.")
    
except Exception as e:
    print(f"Error loading model: {e}")

# try:
#     #model = ChatOpenAI(model=model_name, temperature=0)
#     model = ChatOllama(model=model_name, temperature=0.6)
#     print(f"\nModel {model_name} loaded successfully.")
# except Exception as e:
#     print(f"Error loading model: {e}")

# Config Settings
config = {
    'configurable': {
        'model': model, 
        'model_name': model_name, 
        #'thread_id': 'demo_1',
        'total_attempts': 5,          # total 
        'attempts_w_filtered': 3,     # with filtered
    },
    'recursion_limit': 50
}

# Setting up inputs for next cell
inputs = {
    'database_url': os.getenv("NEO4J_URI"),           
    'database_name': "movies",                        
    'database_user': os.getenv("NEO4J_USERNAME"),    
    'database_password': os.getenv("NEO4J_PASSWORD")  
}

print("\nSetup Complete! Ready for our agent!")

In [ ]:
custom_dataset = [
    "Find all actors who acted in more than 3 movies",
    "Find actors and how many distinct movies they acted in",
    "Find movies that have more than 3 actors",
    "Find actors who acted in movies directed by Ron Howard",
    "Find actors who acted in exactly one movie",
    "Find all directors who directed at least 2 movies",
    "Find all people who both acted in and directed movies",
    "Find all actors who acted in The Matrix",
    "Find how many movies were released per year",
    "Return each movie along with its director"
]

print("Starting evaluation for custom_dataset on Movies DB...")
results_log = [] 

for i, question in enumerate(tqdm(custom_dataset, desc="Evaluating Queries")):
    print('\n' + '#' * 60)
    print(f" QUERY {i+1}: {question}")
    print('#' * 60)
    
    inputs['question'] = question
    
    try:
        result = app.invoke(inputs, config=config)
        answer = result.get('answer')
        
        print('\n' + '='*50)
        print(f"Final Agent Answer: {answer}")
        print('='*50 + '\n')
        
        results_log.append({"question": question, "status": "Success", "answer": answer})
        
    except Exception as e:
        print(f"\n Execution Failed: {e}")
        results_log.append({"question": question, "status": "Failed", "error": str(e)})
    
    if i < len(custom_dataset) - 1:
        print("Waiting 10s for API rate limits...")
        time.sleep(10)

print("\n Evaluation Complete!")